# 🧪 InvRender (Modeling Indirect Illumination for Inverse Rendering) - TensoIR Lego Test

This notebook runs **InvRender (CVPR 2022)** Novel View Synthesis (NVS) testing on the **TensoIR Lego dataset**.

In [ ]:
# 1. Install aria2 & dependencies
!apt-get update -y && apt-get install -y aria2
!pip install -q GPUtil pyhocon tensorboardX ipdb imageio imageio-ffmpeg opencv-python gdown lpips
!nvidia-smi

In [ ]:
# 2. Clone InvRender repository
%cd /content
!git clone https://github.com/zju3dv/InvRender.git /content/InvRender || true
%cd /content/InvRender

In [ ]:
# 3. Fast 16-thread download of TensoIR Lego dataset from Zenodo via aria2c
!mkdir -p /content/InvRender/data
%cd /content/InvRender/data
!aria2c -x 16 -s 16 -k 1M -o lego.zip "https://zenodo.org/records/7880113/files/lego.zip?download=1"
!unzip -q lego.zip -d lego
!rm -f lego.zip
%cd /content/InvRender

In [ ]:
# 4. Append TensoIRDataset directly to syn_dataset.py & generate config
import os, sys, importlib

dataset_code = '''

# =========================================================
# Custom TensoIRDataset for TensoIR Lego Dataset
# =========================================================
class TensoIRDataset(torch.utils.data.Dataset):
    def __init__(self, instance_dir, frame_skip=1, split='train'):
        self.instance_dir = instance_dir
        self.split = split
        print(f'Creating TensoIRDataset from: {self.instance_dir} (split: {split})')
        
        target_dir = os.path.join(self.instance_dir, split)
        if not os.path.exists(target_dir):
            target_dir = self.instance_dir
            
        subdirs = sorted([d for d in os.listdir(target_dir) if os.path.isdir(os.path.join(target_dir, d))])
        subdirs = subdirs[::frame_skip]
        
        self.n_cameras = len(subdirs)
        self.rgb_images = []
        self.object_masks = []
        self.intrinsics_all = []
        self.pose_all = []
        self.image_paths = []
        scale = 2.0
        
        for d in subdirs:
            d_path = os.path.join(target_dir, d)
            jsons = [f for f in os.listdir(d_path) if f.endswith('.json')]
            if not jsons:
                continue
            with open(os.path.join(d_path, jsons[0]), 'r') as f:
                meta = json.load(f)
                
            img_path = None
            for cand in ['rgba.png', 'rgb.png', 'image.png', f'{d}.png']:
                p = os.path.join(d_path, cand)
                if os.path.exists(p):
                    img_path = p
                    break
            if img_path is None:
                pngs = [f for f in os.listdir(d_path) if f.endswith('.png') and f not in ['albedo.png', 'normal.png', 'depth.png', 'roughness.png']]
                if pngs:
                    img_path = os.path.join(d_path, pngs[0])
            if img_path is None:
                continue
                
            img = Image.open(img_path)
            W, H = img.size
            
            if 'cam_K' in meta:
                K = np.array(meta['cam_K']).reshape(3, 3).astype(np.float32)
                R_w2c = np.array(meta['cam_R']).reshape(3, 3)
                T_w2c = np.array(meta['cam_T']).flatten()
                w2c = np.eye(4)
                w2c[:3, :3] = R_w2c
                w2c[:3, 3] = T_w2c
                c2w = np.linalg.inv(w2c)
            elif 'cam_transform_mat' in meta:
                val = meta['cam_transform_mat']
                c2w = np.fromstring(val, sep=',').reshape(4, 4) if isinstance(val, str) else np.array(val).reshape(4, 4)
                K = np.array([[1.2*W, 0, W/2], [0, 1.2*W, H/2], [0, 0, 1]], dtype=np.float32)
            elif 'transform_matrix' in meta:
                c2w = np.array(meta['transform_matrix'])
                angle_x = meta.get('camera_angle_x', 0.8)
                focal = 0.5 * W / np.tan(0.5 * angle_x)
                K = np.array([[focal, 0, W/2], [0, focal, H/2], [0, 0, 1]], dtype=np.float32)
            else:
                c2w = np.eye(4)
                K = np.array([[1.2*W, 0, W/2], [0, 1.2*W, H/2], [0, 0, 1]], dtype=np.float32)
                
            c2w[:3, 3] /= scale
            
            im_rgba = np.array(img.convert('RGBA')).astype(np.float32) / 255.0
            rgb = im_rgba[:, :, :3] * im_rgba[:, :, 3:4] + (1.0 - im_rgba[:, :, 3:4])
            mask = im_rgba[:, :, 3] > 0.5
            
            self.rgb_images.append(torch.from_numpy(rgb.reshape(-1, 3)).float())
            self.object_masks.append(torch.from_numpy(mask.reshape(-1)).bool())
            self.intrinsics_all.append(torch.from_numpy(K).float())
            self.pose_all.append(torch.from_numpy(c2w).float())
            self.image_paths.append(img_path)
            
        self.img_res = [H, W]
        self.total_pixels = H * W
        self.has_groundtruth = True

    def __len__(self):
        return self.n_cameras

    def __getitem__(self, idx):
        uv = np.mgrid[0:self.img_res[0], 0:self.img_res[1]]
        uv = torch.from_numpy(uv).float().reshape(2, -1).transpose(1, 0)
        return idx, {
            'object_mask': self.object_masks[idx],
            'uv': uv,
            'intrinsics': self.intrinsics_all[idx],
            'pose': self.pose_all[idx],
            'rgb': self.rgb_images[idx]
        }
'''

syn_file = '/content/InvRender/code/datasets/syn_dataset.py'
with open(syn_file, 'r') as f:
    existing = f.read()

if 'TensoIRDataset' not in existing:
    with open(syn_file, 'a') as f:
        f.write(dataset_code)

conf_code = '''train{
    expname = lego_tensoir_nvs
    dataset_class = datasets.syn_dataset.TensoIRDataset
    model_class = model.implicit_differentiable_renderer.IDRNetwork
    loss_class = model.loss.InvLoss
    illum_loss_class = model.loss.IllumLoss
    plot_freq = 1000
    ckpt_freq = 2000
    num_pixels = 1024
    illum_num_pixels = 256
    alpha_milestones = [25000,50000,75000,100000,125000]
    alpha_factor = 2
    idr_learning_rate = 1e-4
    idr_sched_milestones = [100000,150000]
    idr_sched_factor = 0.5
    idr_epoch = 2000
}
loss{
    idr_rgb_weight = 1.0
    eikonal_weight = 0.1
    mask_weight = 100.0
    alpha = 50.0
    sg_rgb_weight = 1.0
    kl_weight = 0.01
    latent_smooth_weight = 0.1
    loss_type = L1
}
model{
    feature_vector_size = 256
    implicit_network{
        d_in = 3
        d_out = 1
        dims = [ 512, 512, 512, 512, 512, 512, 512, 512 ]
        geometric_init = True
        bias = 0.6
        skip_in = [4]
        weight_norm = True
        multires = 6
    }
    rendering_network{
        d_in = 9
        d_out = 3
        dims = [ 512, 512, 512, 512 ]
        weight_norm = True
        multires_view = 4
    }
    ray_tracer{
        object_bounding_sphere = 1.0
        sdf_threshold = 5.0e-5
        line_search_tolerance = 1.0e-5
        sphere_tracing_gss_factor = 0.5
        n_steps = 100
        n_secant_steps = 8
    }
}
'''
with open('/content/InvRender/code/confs_sg/lego_tensoir.conf', 'w') as f:
    f.write(conf_code)
print('TensoIRDataset attached to syn_dataset.py & config created successfully!')

In [ ]:
# 5. Run InvRender Stage 1 IDR NVS Training
%cd /content/InvRender/code
!python3 training/exp_runner.py --conf confs_sg/lego_tensoir.conf --data_split_dir ../data/lego --expname lego_tensoir_nvs --trainstage IDR --gpu 0